# **💁🏻🗨️💁🏻‍♂️대화 요약 SOLAR API code**
> **Dialogue Summarization** 경진대회에 오신 여러분 환영합니다! 🎉    
> 본 자료에서는 Solar Chat API를 이용하여 대화 요약 대회를 풀어봅니다.     

## ⚙️ 데이터 및 환경설정

### 1) 필요한 라이브러리 설치

- 필요한 라이브러리를 설치한 후 불러옵니다.

In [1]:
!pip install openai

  Obtaining dependency information for openai from https://files.pythonhosted.org/packages/07/b4/57f1954a4560092ad8c45f07ad183eab9c8e093e0a1db829f9b506b2d5d1/openai-1.59.9-py3-none-any.whl.metadata
  Obtaining dependency information for distro<2,>=1.7.0 from https://files.pythonhosted.org/packages/12/b3/231ffd4ab1fc9d679809f356cebee130ac7daa00d6d6f3206dd4fd137e9e/distro-1.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for httpx<1,>=0.23.0 from https://files.pythonhosted.org/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943ec3f6a384bdc792deb2/httpx-0.28.1-py3-none-any.whl.metadata
  Obtaining dependency information for jiter<1,>=0.4.0 from https://files.pythonhosted.org/packages/4d/a0/3993cda2e267fe679b45d0bcc2cef0b4504b0aa810659cdae9737d6bace9/jiter-0.8.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for pydantic<3,>=1.9.0 from https://files.pythonhosted.org/packages/58/26/82663c79010b28eddf29dcdd0ea72343

In [2]:
import pandas as pd
import os
import time
from tqdm import tqdm
from rouge import Rouge # 모델의 성능을 평가하기 위한 라이브러리입니다.
from openai import OpenAI # openai==1.2.0

### 2) Solar Chat API Client 생성하기
- 앞으로 Solar Chat API를 사용하기 위해 Client를 생성합니다.

In [5]:
UPSTAGE_API_KEY = "up_aiCe2KyaGnLmvgVLham0vcOEudzdc" # upstage.ai에서 발급받은 API KEY를 입력해주세요.

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1/solar"
)

### 3) Solar Chat API 사용해보기 (선택)
- 예시 코드를 통해 Solar Chat API를 사용해보세요.

In [6]:
stream = client.chat.completions.create(
    model="solar-1-mini-chat",
    messages=[
      {
        "role": "system",
        "content": "You are a helpful assistant."
      },
      {
        "role": "user",
        "content": "Hello!"
      }
    ],
    stream=True,
)
 
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")
 
# Use with stream=False
# print(stream.choices[0].message.content)

Hello! How may I assist you today?

### 4) 데이터 불러오기
- 실험에서 쓰일 데이터를 load합니다.

In [7]:
# 데이터 경로를 지정해줍니다.
DATA_PATH = "../data/"
RESULT_PATH = "./prediction/"

# train data의 구조와 내용을 확인합니다.
train_df = pd.read_csv(os.path.join(DATA_PATH,'train.csv'))
train_df.tail()

,fname,dialogue,summary,topic
12452,train_12455,#Person1#: 실례합니다. 맨체스터 출신의 그린 씨이신가요?\n#Person2...,탄 링은 흰머리와 수염으로 쉽게 인식되는 그린 씨를 만나 호텔로 데려갈 예정입니다....,누군가를 태우다
12453,train_12456,#Person1#: 이윙 씨가 우리가 컨퍼런스 센터에 오후 4시에 도착해야 한다고 ...,#Person1#과 #Person2#는 이윙 씨가 늦지 않도록 요청했기 때문에 컨퍼...,컨퍼런스 센터
12454,train_12457,#Person1#: 오늘 어떻게 도와드릴까요?\n#Person2#: 차를 빌리고 싶...,#Person2#는 #Person1#의 도움으로 5일 동안 소형 차를 빌립니다.,차 렌트
12455,train_12458,#Person1#: 오늘 좀 행복해 보이지 않아. 무슨 일 있어?\n#Person2...,#Person2#의 엄마가 일자리를 잃었다. #Person2#는 엄마가 우울해하지 ...,실직
12456,train_12459,"#Person1#: 엄마, 다음 토요일에 이 삼촌네 가족을 방문하기 위해 비행기를 ...",#Person1#은 다음 토요일에 이 삼촌네를 방문할 때 가방을 어떻게 싸야 할지 ...,짐 싸기


In [8]:
# validation data의 구조와 내용을 확인합니다.
val_df = pd.read_csv(os.path.join(DATA_PATH,'dev.csv'))
val_df.tail()

,fname,dialogue,summary,topic
494,dev_495,#Person1#: 이제 새해가 되어서 새로운 시작을 하려고 결심했어. \r\n#P...,#Person1#은 새해에 금연을 하고 커밍아웃하기로 결정했습니다. #Person2...,새해
495,dev_496,"#Person1#: 너, 조랑 결혼했지? \r\n#Person2#: 조? 무슨 말인...",#Person1#은 #Person2#가 조와 결혼했다고 생각했다. #Person2#...,사랑에 빠지다
496,dev_497,"#Person1#: 무엇을 도와드릴까요, 부인?\r\n#Person2#: 몇 주 동...",#Person2#의 차에서 이상한 소리가 납니다. #Person1#는 브레이크를 교...,소음
497,dev_498,"#Person1#: 안녕하세요, 아마존 고객 서비스입니다. 무엇을 도와드릴까요?\n...",#Person2#님이 아마존 고객 서비스에 전화하여 아마존에서 받은 책에 한 페이지...,빠진 페이지
498,dev_499,#Person1#: 여름이 다 되어간다는 게 믿기지 않아.\r\n#Person2#:...,#Person2#는 #Person1#에게 여름 휴가 동안 파티를 도와주는 회사에서 ...,여름 휴가


## 1. Solar Chat API 요약 성능 확인하기
- Solar Chat API을 이용하여 train 및 validation dataset에 포함된 dialogue 샘플을 요약해 봅니다.

In [9]:
# 모델 성능에 대한 평가 지표를 정의합니다. 본 대회에서는 ROUGE 점수를 통해 모델의 성능을 평가합니다.
rouge = Rouge()
def compute_metrics(pred, gold):
    results = rouge.get_scores(pred, gold, avg=True)
    result = {key: value["f"] for key, value in results.items()}
    return result

In [10]:
# Dialogue를 입력으로 받아, Solar Chat API에 보낼 Prompt를 생성하는 함수를 정의합니다.
def build_prompt(dialogue):
    system_prompt = "You are an expert in the field of dialogue summarization. Please summarize the following dialogue."

    user_prompt = f"Dialogue:\n{dialogue}\n\nSummary:\n"
    
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

In [11]:
# Solar Chat API를 활용해 Summarization을 수행하는 함수를 정의합니다.
def summarization(dialogue):
    summary = client.chat.completions.create(
        model="solar-1-mini-chat",
        messages=build_prompt(dialogue),
    )

    return summary.choices[0].message.content

### (선택) parameter 변경하기
- Solar Chat API를 사용할 때, parameter를 변경하여, 다양한 결과를 얻을 수 있습니다.
- Parameter에 대한 자세한 설명은 [여기](https://developers.upstage.ai/docs/apis/chat#request-body)를 참고해주세요.

In [12]:
def summarization(dialogue):
    summary = client.chat.completions.create(
        model="solar-1-mini-chat",
        messages=build_prompt(dialogue),
        temperature=0.2,
        top_p=0.3,
    )

    return summary.choices[0].message.content

Train Dataset을 이용하여 요약이 잘 되는지 확인해 봅니다.

In [4]:
# Train data 중 처음 3개의 대화를 요약합니다.
def test_on_train_data(num_samples=3):
    for idx, row in train_df[:num_samples].iterrows():
        dialogue = row['dialogue']
        summary = summarization(dialogue)
        print(f"Dialogue:\n{dialogue}\n")
        print(f"Pred Summary: {summary}\n")
        print(f"Gold Summary: {row['summary']}\n")
        print("=="*50)

In [14]:
if __name__ == "__main__":
    test_on_train_data()

Dialogue:
#Person1#: 안녕하세요, 스미스씨. 저는 호킨스 의사입니다. 오늘 왜 오셨나요?
#Person2#: 건강검진을 받는 것이 좋을 것 같아서요.
#Person1#: 그렇군요, 당신은 5년 동안 건강검진을 받지 않았습니다. 매년 받아야 합니다.
#Person2#: 알고 있습니다. 하지만 아무 문제가 없다면 왜 의사를 만나러 가야 하나요?
#Person1#: 심각한 질병을 피하는 가장 좋은 방법은 이를 조기에 발견하는 것입니다. 그러니 당신의 건강을 위해 최소한 매년 한 번은 오세요.
#Person2#: 알겠습니다.
#Person1#: 여기 보세요. 당신의 눈과 귀는 괜찮아 보입니다. 깊게 숨을 들이쉬세요. 스미스씨, 담배 피우시나요?
#Person2#: 네.
#Person1#: 당신도 알다시피, 담배는 폐암과 심장병의 주요 원인입니다. 정말로 끊으셔야 합니다. 
#Person2#: 수백 번 시도했지만, 습관을 버리는 것이 어렵습니다.
#Person1#: 우리는 도움이 될 수 있는 수업과 약물들을 제공하고 있습니다. 나가기 전에 더 많은 정보를 드리겠습니다.
#Person2#: 알겠습니다, 감사합니다, 의사선생님.

Pred Summary: 호킨스 의사는 스미스씨에게 매년 건강검진을 받는 것이 심각한 질병을 조기에 발견하여 예방하는 데 중요하다고 조언합니다. 의사는 스미스씨의 눈과 귀를 검사하고, 폐암과 심장병의 위험을 증가시키는 흡연 습관을 끊을 것을 권장합니다. 의사는 스미스씨에게 도움을 줄 수 있는 수업과 약물에 대한 정보를 제공합니다.

Gold Summary: 스미스씨가 건강검진을 받고 있고, 호킨스 의사는 매년 건강검진을 받는 것을 권장합니다. 호킨스 의사는 스미스씨가 담배를 끊는 데 도움이 될 수 있는 수업과 약물에 대한 정보를 제공할 것입니다.

Dialogue:
#Person1#: 안녕하세요, 파커 부인, 어떻게 지내셨나요?
#Person2#: 안녕하세요, 피터스 박사님. 잘 지냈습니다, 감사합니다. 리키와 함께 백신 접종을 

Validation Dataset을 이용하여 요약을 진행하고, 성능을 평가해 봅니다.

In [5]:
# Validation data의 대화를 요약하고, 점수를 측정합니다.
def validate(num_samples=-1):
    val_samples = val_df[:num_samples] if num_samples > 0 else val_df
    
    scores = []
    for idx, row in tqdm(val_samples.iterrows(), total=len(val_samples)):
        dialogue = row['dialogue']
        summary = summarization(dialogue)
        results = compute_metrics(summary, row['summary'])
        avg_score = sum(results.values()) / len(results)
        
        scores.append(avg_score)
        
    val_avg_score = sum(scores) / len(scores)

    print(f"Validation Average Score: {val_avg_score}")

In [16]:
if __name__ == "__main__":
    validate(100) # 100개의 validation sample에 대한 요약을 수행합니다.
    
    # 전체 validation data에 대한 요약을 수행하고 싶은 경우 아래와 같이 실행합니다.
    # validate() 

100%|██████████| 100/100 [02:03<00:00,  1.24s/it]

Validation Average Score: 0.1380123191965196


## 2. Solar Chat API로 요약하기
- Solar Chat API을 이용하여 test dataset에 포함된 dialogue를 요약하고 제출용 파일을 생성합니다.

In [6]:
def inference():
    test_df = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))

    summary = []
    start_time = time.time()
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        dialogue = row['dialogue']
        summary.append(summarization(dialogue))
        
        # Rate limit 방지를 위해 1분 동안 최대 100개의 요청을 보내도록 합니다.
        if (idx + 1) % 100 == 0:
            end_time = time.time()
            elapsed_time = end_time - start_time
            
            if elapsed_time < 60:
                wait_time = 60 - elapsed_time + 5
                print(f"Elapsed time: {elapsed_time:.2f} sec")
                print(f"Waiting for {wait_time} sec")
                time.sleep(wait_time)
            
            start_time = time.time()
    
    output = pd.DataFrame(
        {
            "fname": test_df['fname'],
            "summary" : summary,
        }
    )
    
    if not os.path.exists(RESULT_PATH):
        os.makedirs(RESULT_PATH)
    output.to_csv(os.path.join(RESULT_PATH, "output_solar_new.csv"), index=False)

    return output

In [18]:
if __name__ == "__main__":
    output = inference()

100%|██████████| 499/499 [10:24<00:00,  1.25s/it]


In [19]:
output  # 각 대화문에 대한 요약문이 출력됨을 확인할 수 있습니다.

,fname,summary
0,test_0,"이 대화에서, 실장 #Person1#은 직원 #Person2#에게 새로운 사무실 정..."
1,test_1,#Person1#은 #Person2#가 교통 체증으로 인해 늦게 도착한 것에 대해 ...
2,test_2,마샤와 히어로는 2개월 동안 별거한 후 이혼을 신청했습니다. 마샤가 양육권을 가지게...
3,test_3,"이 대화에서, Person1은 Person2의 생일을 축하하고 파티에 초대합니다. ..."
4,test_4,두 사람이 올림픽 공원의 중심인 올림픽 스타디움에 대해 이야기하고 있습니다. 스타디...
...,...,...
494,test_495,잭은 찰리에게 학교 끝나고 자신의 집에서 비디오 게임을 하자고 제안합니다. 그들은 ...
495,test_496,대화에서 Person2는 아내와 함께 레코드 플레이어를 구입한 후 컨트리 음악에 관...
496,test_497,"한 사람이 세탁기와 건조기를 사용하는 방법을 물어보고, 비누를 사용하는 방법에 대해..."
497,test_498,스티브와 매튜는 오랜만에 만나서 이야기를 나눕니다. 스티브는 매튜가 새로운 집을 찾...


## 3. Prompt Engineering
- Prompt engineering을 통해 요약 성능 향상을 시도합니다.

In [20]:
# Few-shot prompt를 생성하기 위해, train data의 일부를 사용합니다.
few_shot_samples = train_df.sample(1)

sample_dialogue1 = few_shot_samples.iloc[0]['dialogue']
sample_summary1 = few_shot_samples.iloc[0]['summary']

print(f"Sample Dialogue1:\n{sample_dialogue1}\n")
print(f"Sample Summary1: {sample_summary1}\n")

Sample Dialogue1:
#Person1#: 너무 슬퍼하지 마. 만약 정말로 그와의 감정이 없다고 생각한다면, 내 생각에는 이 문제를 해결하는 가장 좋은 방법은 이혼일지도 모르겠어.
#Person2#: 나도 마음속 깊은 곳에서 잘 알고 있어. 그저 아이 때문에 마음을 놓을 수 없어. 그녀는 어려. 우리를 이해하거나 이런 사실을 받아들일 수 없어.
#Person1#: 그래, 아이가 문제야. 진실을 제니에게 말하지 마, 그녀에게는 선의의 거짓말만 해. 그녀가 성장하면 적절한 기회를 찾아서 말해주면 돼.
#Person2#: 알겠어. 좋아.

Sample Summary1: #Person1#은 #Person2#에게 이혼을 권하지만 #Person2#는 딸을 걱정하고 있다. #Person1#은 그녀에게 선의의 거짓말을 하라고 제안한다.



In [21]:
# Prompt를 생성하는 함수를 수정합니다.
def build_prompt(dialogue):
    system_prompt = "You are a expert in the field of dialogue summarization, summarize the given dialogue in a concise manner. Follow the user's instruction carefully and provide a summary that is relevant to the dialogue."

    user_prompt = (
        "Following the instructions below, summarize the given document.\n"
        "Instructions:\n"
        "1. Read the provided sample dialogue and corresponding summary.\n"
        "2. Read the dialogue carefully.\n"
        "3. Following the sample's style of summary, provide a concise summary of the given dialogue.\n\n"
        "Sample Dialogue:\n"
        f"{sample_dialogue1}\n\n"
        "Sample Summary:\n"
        f"{sample_summary1}\n\n"
        "Dialogue:\n"
        f"{dialogue}\n\n"
        "Summary:\n"
    )
    
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

In [22]:
# 변경된 prompt를 사용하여, train data 중 처음 3개의 대화를 요약하고, 결과를 확인합니다.
if __name__ == "__main__":
    test_on_train_data()

Dialogue:
#Person1#: 안녕하세요, 스미스씨. 저는 호킨스 의사입니다. 오늘 왜 오셨나요?
#Person2#: 건강검진을 받는 것이 좋을 것 같아서요.
#Person1#: 그렇군요, 당신은 5년 동안 건강검진을 받지 않았습니다. 매년 받아야 합니다.
#Person2#: 알고 있습니다. 하지만 아무 문제가 없다면 왜 의사를 만나러 가야 하나요?
#Person1#: 심각한 질병을 피하는 가장 좋은 방법은 이를 조기에 발견하는 것입니다. 그러니 당신의 건강을 위해 최소한 매년 한 번은 오세요.
#Person2#: 알겠습니다.
#Person1#: 여기 보세요. 당신의 눈과 귀는 괜찮아 보입니다. 깊게 숨을 들이쉬세요. 스미스씨, 담배 피우시나요?
#Person2#: 네.
#Person1#: 당신도 알다시피, 담배는 폐암과 심장병의 주요 원인입니다. 정말로 끊으셔야 합니다. 
#Person2#: 수백 번 시도했지만, 습관을 버리는 것이 어렵습니다.
#Person1#: 우리는 도움이 될 수 있는 수업과 약물들을 제공하고 있습니다. 나가기 전에 더 많은 정보를 드리겠습니다.
#Person2#: 알겠습니다, 감사합니다, 의사선생님.

Pred Summary: #Person1#은 #Person2#에게 건강검진을 받도록 권장하며, 심각한 질병을 조기에 발견하는 것의 중요성을 강조합니다. #Person2#는 건강검진의 중요성을 인정하지만, 아무 문제가 없다면 왜 의사를 만나야 하는지 의문을 제기합니다. #Person1#은 건강검진의 중요성을 재차 강조하며, #Person2#의 흡연 습관을 언급하고 이를 끊을 것을 권장합니다. #Person2#는 습관을 버리는 것이 어렵다고 인정하며, #Person1#은 도움이 될 수 있는 수업과 약물을 제안합니다.

Gold Summary: 스미스씨가 건강검진을 받고 있고, 호킨스 의사는 매년 건강검진을 받는 것을 권장합니다. 호킨스 의사는 스미스씨가 담배를 끊는 데 도움이 될 수 있는 수업과 약물에 대한 정보를 제공할 것입니다.


In [23]:
# 변경된 prompt를 사용하여, validation data의 대화를 요약하고, 점수를 측정합니다.
if __name__ == "__main__":
    validate(100)

100%|██████████| 100/100 [01:50<00:00,  1.10s/it]

Validation Average Score: 0.1698978721728168


다른 방식으로 Few-shot sample을 제공하여 Prompt를 구성해 봅니다.

In [7]:
# Few-shot sample을 다른 방식으로 사용하여 prompt를 생성합니다.
def build_prompt(dialogue):
    system_prompt = "You are a expert in the field of dialogue summarization, summarize the given dialogue in a concise manner. Follow the user's instruction carefully and provide a summary that is relevant to the dialogue."

    few_shot_user_prompt_1 = (
        "Following the instructions below, summarize the given document.\n"
        "Instructions:\n"
        "1. Read the provided sample dialogue and corresponding summary.\n"
        "2. Read the dialogue carefully.\n"
        "3. Following the sample's style of summary, provide a concise summary of the given dialogue. Be sure that the summary is simple but captures the essence of the dialogue.\n\n"
        "Dialogue:\n"
        f"{sample_dialogue1}\n\n"
        "Summary:\n"
    )
    few_shot_assistant_prompt_1 = sample_summary1
    
    user_prompt = (
        "Dialogue:\n"
        f"{dialogue}\n\n"
        "Summary:\n"
    )
    
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": few_shot_user_prompt_1},
        {"role": "assistant", "content": few_shot_assistant_prompt_1},
        {"role": "user", "content": user_prompt},
    ]

In [25]:
# 변경된 prompt를 사용하여, train data 중 처음 3개의 대화를 요약하고, 결과를 확인합니다.
if __name__ == "__main__":
    test_on_train_data()

Dialogue:
#Person1#: 안녕하세요, 스미스씨. 저는 호킨스 의사입니다. 오늘 왜 오셨나요?
#Person2#: 건강검진을 받는 것이 좋을 것 같아서요.
#Person1#: 그렇군요, 당신은 5년 동안 건강검진을 받지 않았습니다. 매년 받아야 합니다.
#Person2#: 알고 있습니다. 하지만 아무 문제가 없다면 왜 의사를 만나러 가야 하나요?
#Person1#: 심각한 질병을 피하는 가장 좋은 방법은 이를 조기에 발견하는 것입니다. 그러니 당신의 건강을 위해 최소한 매년 한 번은 오세요.
#Person2#: 알겠습니다.
#Person1#: 여기 보세요. 당신의 눈과 귀는 괜찮아 보입니다. 깊게 숨을 들이쉬세요. 스미스씨, 담배 피우시나요?
#Person2#: 네.
#Person1#: 당신도 알다시피, 담배는 폐암과 심장병의 주요 원인입니다. 정말로 끊으셔야 합니다. 
#Person2#: 수백 번 시도했지만, 습관을 버리는 것이 어렵습니다.
#Person1#: 우리는 도움이 될 수 있는 수업과 약물들을 제공하고 있습니다. 나가기 전에 더 많은 정보를 드리겠습니다.
#Person2#: 알겠습니다, 감사합니다, 의사선생님.

Pred Summary: #Person1#은 #Person2#에게 건강검진을 받도록 권유하며, 심각한 질병을 조기에 발견하는 것이 중요하다고 강조한다. #Person1#은 또한 #Person2#에게 흡연을 그만두도록 조언하며, 이를 돕기 위한 수업과 약물을 제공한다.

Gold Summary: 스미스씨가 건강검진을 받고 있고, 호킨스 의사는 매년 건강검진을 받는 것을 권장합니다. 호킨스 의사는 스미스씨가 담배를 끊는 데 도움이 될 수 있는 수업과 약물에 대한 정보를 제공할 것입니다.

Dialogue:
#Person1#: 안녕하세요, 파커 부인, 어떻게 지내셨나요?
#Person2#: 안녕하세요, 피터스 박사님. 잘 지냈습니다, 감사합니다. 리키와 함께 백신 접종을 위해 왔습니다.
#Person1#: 좋습니다. 백신 접종 기록을 

In [26]:
# 변경된 prompt를 사용하여, validation data의 대화를 요약하고, 점수를 측정합니다.
if __name__ == "__main__":
    validate(100)

100%|██████████| 100/100 [01:30<00:00,  1.10it/s]

Validation Average Score: 0.17364074587433304


### (선택) 변경된 Prompt로 test dataset에 대한 요약을 진행합니다.
- 변경된 prompt를 통해 점수가 개선되었다면, test dataset에 대한 요약을 진행하고 제출합니다.

In [27]:
# 변경된 prompt를 사용하여, test data의 대화를 요약하고, 결과를 확인합니다.
if __name__ == "__main__":
    output = inference()

100%|██████████| 499/499 [08:16<00:00,  1.00it/s]


In [28]:
output

,fname,summary
0,test_0,#Person1#은 #Person2#에게 모든 사무실 통신을 이메일과 공식 메모로 ...
1,test_1,#Person1#은 #Person2#가 교통 체증으로 인해 늦게 도착한 것에 대해 ...
2,test_2,"마샤와 히어로가 별거 후 이혼을 신청했다. 마샤가 양육권을 가지게 되며, 집과 주식..."
3,test_3,"#Person1#은 브라이언의 생일을 축하하고, 둘은 춤을 추며 파티를 즐긴다. #..."
4,test_4,"#Person1#은 #Person2#와 함께 올림픽 공원을 구경하며, 완공 예정일과..."
...,...,...
494,test_495,잭은 찰리에게 6시에 아빠를 데리러 가기 전에 2시간 동안 새로운 캐릭터 생성 게임...
495,test_496,#Person2#는 아내와 함께 레코드 플레이어를 구입한 후 컨트리 음악에 관심을 ...
496,test_497,#Person1#은 세탁을 해본 적이 없어서 #Person2#에게 도움을 요청한다....
497,test_498,#Person1#과 #Person2#는 오랜만에 만나서 이야기를 나눈다. #Pers...


###이제부터는 새로운 코드로 진행합니다.

In [1]:
# 1. 먼저 기존 패키지들을 제거
!pip uninstall -y httpx openai

Found existing installation: httpx 0.24.1
Uninstalling httpx-0.24.1:
  Successfully uninstalled httpx-0.24.1
Found existing installation: openai 1.2.0
Uninstalling openai-1.2.0:
  Successfully uninstalled openai-1.2.0


In [3]:
# 2. 특정 버전의 httpx 설치
!pip install httpx==0.24.1

  Obtaining dependency information for httpx==0.24.1 from https://files.pythonhosted.org/packages/ec/91/e41f64f03d2a13aee7e8c819d82ee3aa7cdc484d18c0ae859742597d5aa0/httpx-0.24.1-py3-none-any.whl.metadata
  Using cached httpx-0.24.1-py3-none-any.whl.metadata (7.4 kB)
Using cached httpx-0.24.1-py3-none-any.whl (75 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googletrans 3.1.0a0 requires httpx==0.13.3, but you have httpx 0.24.1 which is incompatible.


In [4]:
# 3. 특정 버전의 openai 설치
!pip install openai==1.2.0

  Obtaining dependency information for openai==1.2.0 from https://files.pythonhosted.org/packages/42/9d/b227c396c6eafbece9a8fe877379582481b108c0c0cf4ff8770ce63926ad/openai-1.2.0-py3-none-any.whl.metadata
  Using cached openai-1.2.0-py3-none-any.whl.metadata (16 kB)
Using cached openai-1.2.0-py3-none-any.whl (219 kB)


In [5]:
# 4. 필요한 라이브러리 임포트
import pandas as pd
import os
import time
from tqdm import tqdm
from rouge import Rouge
from openai import OpenAI

In [6]:
UPSTAGE_API_KEY = "up_aiCe2KyaGnLmvgVLham0vcOEudzdc" # upstage.ai에서 발급받은 API KEY를 입력해주세요.

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1/solar"
)

In [7]:
# 데이터 경로를 지정해줍니다.
DATA_PATH = "/home/fine_data/"
RESULT_PATH = "./prediction/"

# train data의 구조와 내용을 확인합니다.
train_df = pd.read_csv(os.path.join(DATA_PATH,'fine_train_llm.csv'))
train_df.tail()

,fname,dialogue,summary,topic
12441,train_12455,#Person1#: 실례합니다. 맨체스터 출신의 그린 씨이신가요?\n#Person2...,탄 링은 흰머리와 수염으로 쉽게 인식되는 그린 씨를 만나 호텔로 데려갈 예정입니다....,누군가를 태우다
12442,train_12456,다음은 정제된 텍스트입니다:\n\n#Person1#: 이윙 씨가 우리가 컨퍼런스 센...,다음과 같이 수정된 텍스트입니다:\n\nPerson1과 Person2는 이윙 씨가 ...,컨퍼런스 센터
12443,train_12457,다음은 정제된 텍스트입니다:\n\n#Person1#: 오늘 어떻게 도와드릴까요?\n...,다음과 같이 수정된 텍스트입니다:\n\nPerson2는 Person1의 도움으로 5...,차 렌트
12444,train_12458,#Person1#: 오늘 좀 행복해 보이지 않아. 무슨 일 있어?\n#Person2...,#Person2#의 엄마가 일자리를 잃었다. #Person2#는 엄마가 우울해하지 ...,실직
12445,train_12459,"#Person1#: 엄마, 다음 토요일에 이 삼촌네 가족을 방문하기 위해 비행기를 ...",#Person1#은 다음 토요일에 이 삼촌네를 방문할 때 가방을 어떻게 싸야 할지 ...,짐 싸기


In [8]:
# validation data의 구조와 내용을 확인합니다.
val_df = pd.read_csv(os.path.join(DATA_PATH,'dev.csv'))
val_df.tail()

,fname,dialogue,summary,topic
494,dev_495,#Person1#: 이제 새해가 되어서 새로운 시작을 하려고 결심했어. \r\n#P...,#Person1#은 새해에 금연을 하고 커밍아웃하기로 결정했습니다. #Person2...,새해
495,dev_496,"#Person1#: 너, 조랑 결혼했지? \r\n#Person2#: 조? 무슨 말인...",#Person1#은 #Person2#가 조와 결혼했다고 생각했다. #Person2#...,사랑에 빠지다
496,dev_497,"#Person1#: 무엇을 도와드릴까요, 부인?\r\n#Person2#: 몇 주 동...",#Person2#의 차에서 이상한 소리가 납니다. #Person1#는 브레이크를 교...,소음
497,dev_498,"#Person1#: 안녕하세요, 아마존 고객 서비스입니다. 무엇을 도와드릴까요?\n...",#Person2#님이 아마존 고객 서비스에 전화하여 아마존에서 받은 책에 한 페이지...,빠진 페이지
498,dev_499,#Person1#: 여름이 다 되어간다는 게 믿기지 않아.\r\n#Person2#:...,#Person2#는 #Person1#에게 여름 휴가 동안 파티를 도와주는 회사에서 ...,여름 휴가


In [9]:
# 모델 성능에 대한 평가 지표를 정의합니다. 본 대회에서는 ROUGE 점수를 통해 모델의 성능을 평가합니다.
rouge = Rouge()
def compute_metrics(pred, gold):
    results = rouge.get_scores(pred, gold, avg=True)
    result = {key: value["f"] for key, value in results.items()}
    return result

In [10]:
# Few-shot prompt를 생성하기 위해, train data의 일부를 사용합니다.
few_shot_samples = train_df.sample(1)

sample_dialogue1 = few_shot_samples.iloc[0]['dialogue']
sample_summary1 = few_shot_samples.iloc[0]['summary']

print(f"Sample Dialogue1:\n{sample_dialogue1}\n")
print(f"Sample Summary1: {sample_summary1}\n")

Sample Dialogue1:
다음은 정제된 텍스트입니다:

#Person1#: 안녕, 아직 사무실에 있어? 벌써 7시야.
#Person2#: 가고 싶지만 아주 중요한 발표를 마무리해야 돼. 내일 아침 회의에서 사장님이 필요하다고 하시고, 나는 이 오후 늦게야 중요한 정보를 전달받았어.
#Person1#: 우리 사장님 답네. 항상 중요한 정보를 늦게 주는 건 특징이야. 도와줄 수 있는 거 있어?
#Person2#: 오, 그럼 좋겠다. 정말 고마워. 이름 목록을 다시 확인해 주면 좋겠어. 모두 정확한지 확인해야 해.
#Person1#: 알았어, 먼저 커피를 내릴까?
#Person2#: 나는 괜찮아. 이미 늦은 시간이야. 이렇게 늦게 커피를 마시면 잠이 안 와.

Sample Summary1: #Person2#는 발표를 마무리하기 위해 초과 근무 중이다. #Person1#은 이름 목록을 다시 확인하는 것을 제안한다.



In [11]:
# Few-shot sample을 다른 방식으로 사용하여 prompt를 생성합니다.
def build_prompt(dialogue):
    system_prompt = (
        "You are an expert in dialogue summarization. Your task is to produce a concise and coherent summary of the given dialogue.\n"
        "Ensure your summary:\n"
        "1. Maintains factual consistency (no fabricated details or omissions of critical events).\n"
        "2. Provides relevant background or context where necessary, so the reader understands the situation.\n"
        "3. Focuses on actions, decisions, and outcomes, explaining their significance.\n"
        "4. Avoids subjective or emotional language. Use direct, neutral descriptions of events in a formal tone.\n"
        "5. Is concise, avoiding unnecessary repetition or excessive detail.\n"
        "6. Preserves logical flow (the summary should read naturally and stay on-topic).\n"
        "7. Uses clear, natural language without random insertions or mistranslations.\n\n"
        "Your output will be evaluated using ROUGE metrics. Capture the essence of the original dialogue accurately and briefly."
    )

    few_shot_user_prompt_1 = (
        "Please follow the instructions below to summarize the given dialogue.\n\n"
        "Instructions (perform each step in order):\n"
        "STEP 1. Read the sample dialogue and its summary to understand the target style.\n"
        "STEP 2. Carefully read the new dialogue provided.\n"
        "STEP 3. Summarize the new dialogue while:\n"
        "   (1) Including all essential information about the core issues, conflicts, or decisions.\n"
        "   (2) Providing any necessary background or context so the actions and outcomes are clear.\n"
        "   (3) Maintaining an objective, formal tone without emotional or subjective judgments.\n"
        "   (4) Ensuring the entire summary does not exceed 105 characters (in total).\n"
        "STEP 4. Recheck the summary to confirm:\n"
        "   - It is factually consistent and logically flows.\n"
        "   - It omits trivial or repetitive details.\n"
        "   - It respects the 105-character limit.\n\n"
        "Remember, your summary will be evaluated using ROUGE metrics, so it must reflect the important points of the dialogue accurately.\n\n"
        "Dialogue:\n"
        f"{sample_dialogue1}\n\n"
        "Summary:\n"
    )
    few_shot_assistant_prompt_1 = sample_summary1
    
    user_prompt = (
        "Please follow the instructions below to summarize the given dialogue.\n\n"
        "Instructions (perform each step in order):\n"
        "STEP 1. Read the sample dialogue and its summary to understand the target style.\n"
        "STEP 2. Carefully read the new dialogue provided.\n"
        "STEP 3. Summarize the new dialogue while:\n"
        "   (1) Including all essential information about the core issues, conflicts, or decisions.\n"
        "   (2) Providing any necessary background or context so the actions and outcomes are clear.\n"
        "   (3) Maintaining an objective, formal tone without emotional or subjective judgments.\n"
        "   (4) Ensuring the entire summary does not exceed 105 characters (in total).\n"
        "STEP 4. Recheck the summary to confirm:\n"
        "   - It is factually consistent and logically flows.\n"
        "   - It omits trivial or repetitive details.\n"
        "   - It respects the 105-character limit.\n\n"
        "Remember, your summary will be evaluated using ROUGE metrics, so it must reflect the important points of the dialogue accurately.\n\n"
        "Dialogue:\n"
        f"{dialogue}\n\n"
        "Summary:\n"
    )
    
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": few_shot_user_prompt_1},
        {"role": "assistant", "content": few_shot_assistant_prompt_1},
        {"role": "user", "content": user_prompt},
    ]

In [46]:
def summarization(dialogue):
    summary = client.chat.completions.create(
        model="solar-pro",
        messages=build_prompt(dialogue),
        temperature=0.15,
        top_p=0.25,
    )

    return summary.choices[0].message.content

In [37]:
# Train data 중 처음 3개의 대화를 요약합니다.
def test_on_train_data(num_samples=3):
    for idx, row in train_df[:num_samples].iterrows():
        dialogue = row['dialogue']
        summary = summarization(dialogue)
        print(f"Dialogue:\n{dialogue}\n")
        print(f"Pred Summary: {summary}\n")
        print(f"Gold Summary: {row['summary']}\n")
        print("=="*50)

In [38]:
# Validation data의 대화를 요약하고, 점수를 측정합니다.
def validate(num_samples=-1):
    val_samples = val_df[:num_samples] if num_samples > 0 else val_df
    
    scores = []
    for idx, row in tqdm(val_samples.iterrows(), total=len(val_samples)):
        dialogue = row['dialogue']
        summary = summarization(dialogue)
        results = compute_metrics(summary, row['summary'])
        avg_score = sum(results.values()) / len(results)
        
        scores.append(avg_score)
        
    val_avg_score = sum(scores) / len(scores)

    print(f"Validation Average Score: {val_avg_score}")

In [39]:
def inference():
    test_df = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))

    summary = []
    start_time = time.time()
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        dialogue = row['dialogue']
        summary.append(summarization(dialogue))
        
        # Rate limit 방지를 위해 1분 동안 최대 100개의 요청을 보내도록 합니다.
        if (idx + 1) % 100 == 0:
            end_time = time.time()
            elapsed_time = end_time - start_time
            
            if elapsed_time < 60:
                wait_time = 60 - elapsed_time + 5
                print(f"Elapsed time: {elapsed_time:.2f} sec")
                print(f"Waiting for {wait_time} sec")
                time.sleep(wait_time)
            
            start_time = time.time()
    
    output = pd.DataFrame(
        {
            "fname": test_df['fname'],
            "summary" : summary,
        }
    )
    
    if not os.path.exists(RESULT_PATH):
        os.makedirs(RESULT_PATH)
    output.to_csv(os.path.join(RESULT_PATH, "output_solar_pro_llm.csv"), index=False)

    return output

In [47]:
# 변경된 prompt를 사용하여, train data 중 처음 3개의 대화를 요약하고, 결과를 확인합니다.
if __name__ == "__main__":
    test_on_train_data()

Dialogue:
다음 텍스트를 정제했습니다:

#Person1#: 안녕하세요, 스미스씨. 저는 호킨스 의사입니다. 오늘 왜 오셨나요?
#Person2#: 건강검진을 받는 것이 좋을 것 같아서요.
#Person1#: 그렇군요, 당신은 5년 동안 건강검진을 받지 않았습니다. 매년 받아야 합니다.
#Person2#: 알고 있습니다. 하지만 아무 문제가 없다면 왜 의사를 만나러 가야 하나요?
#Person1#: 심각한 질병을 피하는 가장 좋은 방법은 이를 조기에 발견하는 것입니다. 그러니 당신의 건강을 위해 최소한 매년 한 번은 오세요.
#Person2#: 알겠습니다.
#Person1#: 여기 보세요. 당신의 눈과 귀는 괜찮아 보입니다. 깊게 숨을 들이쉬세요. 스미스씨, 담배 피우시나요?
#Person2#: 네.
#Person1#: 당신도 알다시피, 담배는 폐암과 심장병의 주요 원인입니다. 정말로 끊으셔야 합니다.
#Person2#: 수백 번 시도했지만, 습관을 버리는 것이 어렵습니다.
#Person1#: 우리는 도움이 될 수 있는 수업과 약물을 제공하고 있습니다. 나가기 전에 더 많은 정보를 드리겠습니다.
#Person2#: 알겠습니다, 감사합니다, 의사선생님.

Pred Summary: #Person1#은 #Person2#에게 매년 건강검진을 받을 것을 권고한다. #Person1#은 또한 #Person2#에게 담배 피우기를 중단할 것을 권고한다.

Gold Summary: 스미스씨가 건강검진을 받고 있으며, 호킨스 의사는 매년 건강검진을 받는 것을 권장합니다. 호킨스 의사는 스미스씨가 담배를 끊는 데 도움이 될 수 있는 수업과 약물에 대한 정보를 제공할 것입니다.

Dialogue:
다음 텍스트를 정제했습니다:

#Person1#: 안녕하세요, 파커 부인, 어떻게 지내셨나요?
#Person2#: 안녕하세요, 피터스 박사님. 잘 지냈습니다, 감사합니다. 리키와 함께 백신 접종을 위해 왔습니다.
#Person1#: 좋습니다. 백신 접종 기록을 보니, 리키는 이미 소

In [48]:
# 변경된 prompt를 사용하여, validation data의 대화를 요약하고, 점수를 측정합니다.
if __name__ == "__main__":
    validate(5)

100%|██████████| 5/5 [00:03<00:00,  1.27it/s]

Validation Average Score: 0.151146719112536


In [50]:
# 변경된 prompt를 사용하여, test data의 대화를 요약하고, 결과를 확인합니다.
if __name__ == "__main__":
    output = inference()

  0%|          | 0/499 [00:00<?, ?it/s]

100%|██████████| 499/499 [10:59<00:00,  1.32s/it]


In [51]:
def validate_summary(summary):
    """
    주어진 요약문이 프롬프트의 요구사항을 충족하는지 확인하는 함수
    """
    # 1. 길이 제한 확인 (105자)
    if len(summary) > 105:
        return False, "Length exceeds 105 characters"
    
    # 2. 기본적인 형식 검증
    if not summary or summary.isspace():
        return False, "Empty or whitespace summary"
        
    # 3. Person1, Person2 형식 확인
    if "#Person" in summary and not all(x in summary for x in ["#Person1", "#Person2"]):
        return False, "Inconsistent person formatting"
    
    return True, "Valid summary"

def refine_with_solar(original_summary, dialogue):
    """
    Solar LLM을 사용하여 요약문을 개선하는 함수
    """
    system_prompt = (
        "You are a dialogue summary reviewer and improver. "
        "Review the given summary and improve it in korean according to these rules:\n"
        "1. Keep it under 105 characters\n"
        "2. Maintain factual accuracy\n"
        "3. Use consistent #Person1#/#Person2# formatting\n"
        "4. Focus on key actions and outcomes\n"
        "5. Use formal, objective language"
    )
    
    user_prompt = (
        f"Original dialogue:\n{dialogue}\n\n"
        f"Current summary:\n{original_summary}\n\n"
        "Please provide an improved summary following the rules above."
    )
    
    try:
        response = client.chat.completions.create(
            model="solar-pro",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.15,
            top_p=0.25
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error in refining summary: {e}")
        return original_summary

In [52]:
def process_output_with_validation(output_df):
    """
    출력 데이터프레임을 검증하고 필요한 경우 개선하는 함수
    """
    improved_summaries = []
    
    for idx, row in tqdm(output_df.iterrows(), total=len(output_df)):
        summary = row['summary']
        is_valid, message = validate_summary(summary)
        
        if not is_valid:
            #print(f"\nInvalid summary found in {row['fname']}: {message}")
            #print(f"Original summary: {summary}")
            
            # Solar LLM으로 요약 개선
            improved_summary = refine_with_solar(summary, row['summary'])
            #print(f"Improved summary: {improved_summary}")
            improved_summaries.append(improved_summary)
        else:
            improved_summaries.append(summary)
    
    # 개선된 요약으로 새로운 데이터프레임 생성
    improved_output = pd.DataFrame({
        'fname': output_df['fname'],
        'summary': improved_summaries
    })
    
    # 결과 저장
    improved_output.to_csv(os.path.join(RESULT_PATH, "output_solar_pro_llm_improved.csv"), index=False)
    
    return improved_output

In [53]:
# 사용 예시
if __name__ == "__main__":
    
    
    # 검증 및 개선 프로세스 실행
    improved_output = process_output_with_validation(output)
    
    # 결과 확인
    print("\nValidation and improvement process completed")
    print(f"Total processed items: {len(improved_output)}")

  0%|          | 0/499 [00:00<?, ?it/s]

100%|██████████| 499/499 [02:12<00:00,  3.76it/s]


Validation and improvement process completed
Total processed items: 499


In [ ]:
def summarization(dialogue):
    summary = client.chat.completions.create(
        model="solar-1-mini-chat",
        messages=build_prompt(dialogue),
        temperature=0.2,
        top_p=0.3,
    )

    return summary.choices[0].message.content

In [ ]:
# Train data 중 처음 3개의 대화를 요약합니다.
def test_on_train_data(num_samples=3):
    for idx, row in train_df[:num_samples].iterrows():
        dialogue = row['dialogue']
        summary = summarization(dialogue)
        print(f"Dialogue:\n{dialogue}\n")
        print(f"Pred Summary: {summary}\n")
        print(f"Gold Summary: {row['summary']}\n")
        print("=="*50)

In [ ]:
if __name__ == "__main__":
    test_on_train_data()